<a href="https://colab.research.google.com/github/datacentertugaskuliah-coder/PraktekJST/blob/main/IHSG_Return_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# =====================================================================
# End-to-End Pipeline: IHSG Return Forecasting (Revisi Final)
# =====================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Flatten, Dropout, Conv1D,
    BatchNormalization, GlobalAveragePooling1D,
    SimpleRNN, LSTM, Input
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import joblib
import os, warnings
warnings.filterwarnings("ignore")
tf.get_logger().setLevel('ERROR')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ===================== DATA ACQUISITION =====================
TICKER_IHSG = "^JKSE"
TICKER_SP500 = "^GSPC"
START_DATE = "2021-01-01"
END_DATE = "2025-12-31"

ihsg = yf.download(TICKER_IHSG, start=START_DATE, end=END_DATE, progress=False)
if isinstance(ihsg.columns, pd.MultiIndex):
    ihsg.columns = ihsg.columns.get_level_values(0)
ihsg = ihsg[["Open","High","Low","Close","Volume"]].dropna()
ihsg.index = pd.to_datetime(ihsg.index)

sp500_raw = yf.download(TICKER_SP500, start=START_DATE, end=END_DATE, progress=False)
if isinstance(sp500_raw.columns, pd.MultiIndex):
    sp500_raw.columns = sp500_raw.columns.get_level_values(0)
sp500 = sp500_raw["Close"].dropna()
sp500.index = pd.to_datetime(sp500.index)
sp500.name = "SP500"

df = ihsg.join(sp500, how="outer")
df.ffill(inplace=True)
df.dropna(inplace=True)

print(f"✅ Data shape: {df.shape} | {df.index[0].date()} → {df.index[-1].date()}")

# ===================== FEATURE ENGINEERING =====================
df["Return_1"] = df["Close"].pct_change()
df["LogReturn_1"] = np.log(df["Close"] / df["Close"].shift(1))
df["Range"] = (df["High"] - df["Low"]) / df["Close"]
df["SP500_Ret"] = df["SP500"].pct_change()

for w in [5,10,20,60]:
    df[f"MA{w}"] = df["Close"].rolling(w).mean()

df["STD10"] = df["Close"].rolling(10).std()
df["STD20"] = df["Close"].rolling(20).std()
df["MOM5"]  = df["Close"] / df["Close"].shift(5) - 1
df["MOM10"] = df["Close"] / df["Close"].shift(10) - 1
df["VolChg"]= df["Volume"].pct_change()

delta = df["Close"].diff()
gain = delta.clip(lower=0); loss = -delta.clip(upper=0)
avg_gain = gain.rolling(14).mean(); avg_loss = loss.rolling(14).mean()
rs = avg_gain / (avg_loss + 1e-12)
df["RSI14"] = 100 - 100/(1+rs)
ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["MACD"] = ema12 - ema26
df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
df["Target"] = np.log(df["Close"].shift(-1) / df["Close"])

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

features = [
    "Open","High","Low","Close","Volume",
    "SP500","SP500_Ret",
    "Return_1","LogReturn_1","Range",
    "MA5","MA10","MA20","MA60",
    "STD10","STD20","MOM5","MOM10",
    "VolChg","RSI14","MACD","MACD_Signal"
]
target_col = "Target"
print(f"Engineered dataset: {df.shape[0]} samples, {len(features)} features")

# ===================== SPLIT & SCALING =====================
n = len(df)
train_end = int(n * 0.8)
test_end  = int(n * 0.9)

train_val_df = df.iloc[:train_end]
test_df      = df.iloc[train_end:test_end]
deploy_df    = df.iloc[test_end:]

feat_scaler = StandardScaler()
feat_scaler.fit(train_val_df[features])
target_scaler = StandardScaler()
target_scaler.fit(train_val_df[[target_col]])

X_all = feat_scaler.transform(df[features])
y_all = target_scaler.transform(df[[target_col]]).flatten()

LOOKBACK = 30

def create_sequences(X, y, L):
    Xs, ys = [], []
    for i in range(len(X)-L):
        Xs.append(X[i:i+L])
        ys.append(y[i+L])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_all, y_all, LOOKBACK)
dates_seq = df.index[LOOKBACK:]

train_mask = dates_seq < df.index[train_end]
test_mask  = (dates_seq >= df.index[train_end]) & (dates_seq < df.index[test_end])
deploy_mask= dates_seq >= df.index[test_end]

X_train, y_train = X_seq[train_mask], y_seq[train_mask]
X_test,  y_test  = X_seq[test_mask],  y_seq[test_mask]
X_deploy, y_deploy = X_seq[deploy_mask], y_seq[deploy_mask]
dates_deploy = dates_seq[deploy_mask]
print(f"Train: {X_train.shape}, Test: {X_test.shape}, Deployment: {X_deploy.shape}")

# ===================== MODELS =====================
def build_mlp(shape):
    model = Sequential([
        Input(shape=shape), Flatten(),
        Dense(128, activation='relu'), Dropout(0.2),
        Dense(64, activation='relu'), Dropout(0.15),
        Dense(32, activation='relu'), Dense(1)
    ], name="MLP")
    model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
    return model

def build_cnn(shape):
    model = Sequential([
        Input(shape=shape),
        Conv1D(64, 3, activation='relu', padding='causal'), BatchNormalization(),
        Conv1D(32, 3, activation='relu', padding='causal'), GlobalAveragePooling1D(),
        Dense(64, activation='relu'), Dropout(0.2), Dense(1)
    ], name="CNN")
    model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
    return model

def build_rnn(shape):
    model = Sequential([
        Input(shape=shape),
        SimpleRNN(64, activation='tanh'), Dropout(0.2),
        Dense(32, activation='relu'), Dense(1)
    ], name="RNN")
    model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
    return model

def build_lstm(shape):
    model = Sequential([
        Input(shape=shape),
        LSTM(64, return_sequences=True), Dropout(0.2),
        LSTM(32, return_sequences=False), Dropout(0.2),
        Dense(16, activation='relu'), Dense(1)
    ], name="LSTM")
    model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
    return model

model_builders = {"MLP": build_mlp, "CNN": build_cnn, "RNN": build_rnn, "LSTM": build_lstm}

# ===================== TRAINING & EVALUATION =====================
EPOCHS = 60
BATCH_SIZE = 32
results, predictions = {}, {}

y_test_actual = target_scaler.inverse_transform(y_test.reshape(-1,1)).flatten()
baseline_pred = np.zeros_like(y_test_actual)

def directional_accuracy(y_true, y_pred):
    return np.mean(np.sign(y_true) == np.sign(y_pred))

for name, builder in model_builders.items():
    print(f"\n=== Training {name} ===")
    model = builder((LOOKBACK, X_train.shape[2]))
    es = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, min_delta=1e-5)
    rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)
    model.fit(X_train, y_train, validation_split=0.15,
              epochs=EPOCHS, batch_size=BATCH_SIZE,
              callbacks=[es, rlr], shuffle=False, verbose=1)
    pred_scaled = model.predict(X_test, verbose=0).flatten()
    pred = target_scaler.inverse_transform(pred_scaled.reshape(-1,1)).flatten()
    predictions[name] = pred
    rmse = np.sqrt(mean_squared_error(y_test_actual, pred))
    mae  = mean_absolute_error(y_test_actual, pred)
    da   = directional_accuracy(y_test_actual, pred) * 100
    results[name] = {"RMSE": rmse, "MAE": mae, "DirAcc": da}
    print(f"RMSE={rmse:.6f}, MAE={mae:.6f}, DirAcc={da:.1f}%")

rmse_base = np.sqrt(mean_squared_error(y_test_actual, baseline_pred))
mae_base  = mean_absolute_error(y_test_actual, baseline_pred)
da_base   = directional_accuracy(y_test_actual, baseline_pred) * 100
print(f"\nBaseline (return=0): RMSE={rmse_base:.6f}, MAE={mae_base:.6f}, DirAcc={da_base:.1f}%")

best_model_name = min(results, key=lambda x: results[x]['RMSE'])
print(f"\nBest model: {best_model_name} (RMSE = {results[best_model_name]['RMSE']:.6f})")
model_saved = results[best_model_name]['RMSE'] < rmse_base
if model_saved:
    print("✅ Neural model outperforms baseline.")
else:
    print("⚠️ No neural model beats baseline.")

# ===================== DEPLOYMENT TEST =====================
print(f"\n===== Deployment Test ({len(dates_deploy)} hari terakhir) =====")
if model_saved:
    X_pre = np.concatenate([X_train, X_test])
    y_pre = np.concatenate([y_train, y_test])
    best_model = model_builders[best_model_name]((LOOKBACK, X_pre.shape[2]))
    es = EarlyStopping(monitor='loss', patience=7, restore_best_weights=True, min_delta=1e-5)
    best_model.fit(X_pre, y_pre, epochs=40, batch_size=BATCH_SIZE,
                   callbacks=[es], shuffle=False, verbose=0)
    deploy_pred_scaled = best_model.predict(X_deploy, verbose=0).flatten()
    deploy_pred = target_scaler.inverse_transform(deploy_pred_scaled.reshape(-1,1)).flatten()
else:
    deploy_pred = np.zeros(len(y_deploy))

y_deploy_actual = target_scaler.inverse_transform(y_deploy.reshape(-1,1)).flatten()
deploy_table = pd.DataFrame({
    "Date": dates_deploy,
    "Actual_LogReturn": y_deploy_actual,
    f"{best_model_name}_Pred": deploy_pred,
    "Abs_Error": np.abs(y_deploy_actual - deploy_pred)
})
print(deploy_table.round(6))

# ===================== MODEL PERSISTENCE =====================
class ReturnPredictPipeline:
    def __init__(self, model, feat_scaler, target_scaler, feature_cols, lookback):
        self.model = model
        self.feat_scaler = feat_scaler
        self.target_scaler = target_scaler
        self.feature_cols = feature_cols
        self.lookback = lookback

    def predict(self, df_new):
        tmp = df_new.copy()
        tmp["Return_1"]    = tmp["Close"].pct_change()
        tmp["LogReturn_1"] = np.log(tmp["Close"] / tmp["Close"].shift(1))
        tmp["Range"]       = (tmp["High"] - tmp["Low"]) / tmp["Close"]
        tmp["SP500_Ret"]   = tmp["SP500"].pct_change()
        for w in [5,10,20,60]:
            tmp[f"MA{w}"] = tmp["Close"].rolling(w).mean()
        tmp["STD10"] = tmp["Close"].rolling(10).std()
        tmp["STD20"] = tmp["Close"].rolling(20).std()
        tmp["MOM5"]  = tmp["Close"] / tmp["Close"].shift(5) - 1
        tmp["MOM10"] = tmp["Close"] / tmp["Close"].shift(10) - 1
        tmp["VolChg"]= tmp["Volume"].pct_change()
        delta = tmp["Close"].diff()
        gain = delta.clip(lower=0); loss = -delta.clip(upper=0)
        avg_gain = gain.rolling(14).mean(); avg_loss = loss.rolling(14).mean()
        rs = avg_gain / (avg_loss + 1e-12)
        tmp["RSI14"] = 100 - 100/(1+rs)
        ema12 = tmp["Close"].ewm(span=12, adjust=False).mean()
        ema26 = tmp["Close"].ewm(span=26, adjust=False).mean()
        tmp["MACD"] = ema12 - ema26
        tmp["MACD_Signal"] = tmp["MACD"].ewm(span=9, adjust=False).mean()
        tmp.replace([np.inf, -np.inf], np.nan, inplace=True)
        tmp.dropna(inplace=True)
        if len(tmp) < self.lookback:
            raise ValueError("Not enough data for sequence.")
        seq = tmp[self.feature_cols].iloc[-self.lookback:].values
        seq_scaled = self.feat_scaler.transform(seq).reshape(1, self.lookback, len(self.feature_cols))
        pred_scaled = self.model.predict(seq_scaled, verbose=0)[0,0]
        return self.target_scaler.inverse_transform([[pred_scaled]])[0,0]

if model_saved:
    pipeline = ReturnPredictPipeline(best_model, feat_scaler, target_scaler, features, LOOKBACK)
    joblib.dump(pipeline, "ihsg_lstm_pipeline.joblib")
    print("\n💾 Pipeline saved as 'ihsg_lstm_pipeline.joblib'")
else:
    print("\n⚠️ No neural model outperformed baseline – pipeline not saved.")

print("\n🏁 Execution complete.")

✅ Data shape: (1292, 6) | 2021-01-04 → 2025-12-30
Engineered dataset: 1230 samples, 22 features
Train: (954, 30, 22), Test: (123, 30, 22), Deployment: (123, 30, 22)

=== Training MLP ===
Epoch 1/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 10s 92ms/step - loss: 1.5236 - mae: 0.9496 - val_loss: 1.3415 - val_mae: 0.8593 - learning_rate: 0.0010
Epoch 2/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 1.0408 - mae: 0.7752 - val_loss: 1.4885 - val_mae: 0.9417 - learning_rate: 0.0010
Epoch 3/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 1.0542 - mae: 0.7925 - val_loss: 1.3698 - val_mae: 0.8914 - learning_rate: 0.0010
Epoch 4/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - loss: 0.9014 - mae: 0.7177 - val_loss: 1.3041 - val_mae: 0.8914 - learning_rate: 0.0010
Epoch 5/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - loss: 0.9017 - mae: 0.7155 - val_loss: 1.3023 - val_mae: 0.8913 - learning_rate: 0.0010
Epoch 6/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - loss: 0.8811 - mae: 0.6982 - val_loss: 1.3644 - val_ma

In [7]:
# Cell 1: Install required packages
!pip install -q streamlit pyngrok yfinance tensorflow scikit-learn joblib matplotlib seaborn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 63.8 MB/s eta 0:00:00


In [8]:
# Cell 2: Write Streamlit app to app.py
%%writefile app.py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Flatten, Dropout, Conv1D,
    BatchNormalization, GlobalAveragePooling1D,
    SimpleRNN, LSTM, Input
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import joblib
import streamlit as st
import os, warnings
warnings.filterwarnings("ignore")
tf.get_logger().setLevel('ERROR')

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Fungsi untuk membuat sequence
def create_sequences(X, y, L):
    Xs, ys = [], []
    for i in range(len(X)-L):
        Xs.append(X[i:i+L])
        ys.append(y[i+L])
    return np.array(Xs), np.array(ys)

# Model builders
def build_mlp(shape):
    model = Sequential([
        Input(shape=shape), Flatten(),
        Dense(128, activation='relu'), Dropout(0.2),
        Dense(64, activation='relu'), Dropout(0.15),
        Dense(32, activation='relu'), Dense(1)
    ], name="MLP")
    model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
    return model

def build_cnn(shape):
    model = Sequential([
        Input(shape=shape),
        Conv1D(64, 3, activation='relu', padding='causal'),
        BatchNormalization(),
        Conv1D(32, 3, activation='relu', padding='causal'),
        GlobalAveragePooling1D(),
        Dense(64, activation='relu'), Dropout(0.2),
        Dense(1)
    ], name="CNN")
    model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
    return model

def build_rnn(shape):
    model = Sequential([
        Input(shape=shape),
        SimpleRNN(64, activation='tanh'), Dropout(0.2),
        Dense(32, activation='relu'), Dense(1)
    ], name="RNN")
    model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
    return model

def build_lstm(shape):
    model = Sequential([
        Input(shape=shape),
        LSTM(64, return_sequences=True), Dropout(0.2),
        LSTM(32, return_sequences=False), Dropout(0.2),
        Dense(16, activation='relu'), Dense(1)
    ], name="LSTM")
    model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
    return model

model_builders = {"MLP": build_mlp, "CNN": build_cnn, "RNN": build_rnn, "LSTM": build_lstm}

# Sidebar
st.sidebar.title("IHSG Return Forecasting Dashboard")
st.sidebar.markdown("**Interactive model training & evaluation**")

# Pilihan model
model_name = st.sidebar.selectbox("Select Model", list(model_builders.keys()))

# Rentang data
start_date = st.sidebar.date_input("Start Date", value=pd.to_datetime("2021-01-01"))
end_date = st.sidebar.date_input("End Date", value=pd.to_datetime("today"))

# Tombol utama
train_btn = st.sidebar.button("Train & Evaluate")

# Session state untuk menyimpan objek
if "trained" not in st.session_state:
    st.session_state.trained = False
if "model" not in st.session_state:
    st.session_state.model = None
if "results" not in st.session_state:
    st.session_state.results = None
if "pipeline" not in st.session_state:
    st.session_state.pipeline = None

# Main content
st.title("IHSG Return Forecasting")
st.write("This dashboard trains a neural network to forecast next-day log return of IHSG, using S&P 500 as external feature.")

if train_btn:
    with st.spinner("Downloading data..."):
        # --- Data Acquisition ---
        ihsg = yf.download("^JKSE", start=start_date, end=end_date, progress=False)
        if isinstance(ihsg.columns, pd.MultiIndex):
            ihsg.columns = ihsg.columns.get_level_values(0)
        ihsg = ihsg[["Open","High","Low","Close","Volume"]].dropna()
        ihsg.index = pd.to_datetime(ihsg.index)

        sp500_raw = yf.download("^GSPC", start=start_date, end=end_date, progress=False)
        if isinstance(sp500_raw.columns, pd.MultiIndex):
            sp500_raw.columns = sp500_raw.columns.get_level_values(0)
        sp500 = sp500_raw["Close"].dropna()
        sp500.index = pd.to_datetime(sp500.index)
        sp500.name = "SP500"

        df = ihsg.join(sp500, how="outer")
        df.ffill(inplace=True)
        df.dropna(inplace=True)

    # --- Feature Engineering ---
    df["Return_1"] = df["Close"].pct_change()
    df["LogReturn_1"] = np.log(df["Close"] / df["Close"].shift(1))
    df["Range"] = (df["High"] - df["Low"]) / df["Close"]
    df["SP500_Ret"] = df["SP500"].pct_change()
    for w in [5,10,20,60]:
        df[f"MA{w}"] = df["Close"].rolling(w).mean()
    df["STD10"] = df["Close"].rolling(10).std()
    df["STD20"] = df["Close"].rolling(20).std()
    df["MOM5"]  = df["Close"] / df["Close"].shift(5) - 1
    df["MOM10"] = df["Close"] / df["Close"].shift(10) - 1
    df["VolChg"] = df["Volume"].pct_change()
    delta = df["Close"].diff()
    gain = delta.clip(lower=0); loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(14).mean(); avg_loss = loss.rolling(14).mean()
    rs = avg_gain / (avg_loss + 1e-12)
    df["RSI14"] = 100 - 100/(1+rs)
    ema12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema26 = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD"] = ema12 - ema26
    df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["Target"] = np.log(df["Close"].shift(-1) / df["Close"])
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

    features = [
        "Open","High","Low","Close","Volume",
        "SP500","SP500_Ret",
        "Return_1","LogReturn_1","Range",
        "MA5","MA10","MA20","MA60",
        "STD10","STD20","MOM5","MOM10",
        "VolChg","RSI14","MACD","MACD_Signal"
    ]

    # --- Split ---
    n = len(df)
    if n < 120:
        st.error("Data terlalu sedikit untuk training (minimal ~120 hari).")
        st.stop()
    train_end = int(n * 0.8)
    test_end  = int(n * 0.9)

    train_val_df = df.iloc[:train_end]
    test_df      = df.iloc[train_end:test_end]
    deploy_df    = df.iloc[test_end:]

    # Scalers
    feat_scaler = StandardScaler()
    feat_scaler.fit(train_val_df[features])
    target_scaler = StandardScaler()
    target_scaler.fit(train_val_df[["Target"]])

    X_all = feat_scaler.transform(df[features])
    y_all = target_scaler.transform(df[["Target"]]).flatten()

    LOOKBACK = 30
    X_seq, y_seq = create_sequences(X_all, y_all, LOOKBACK)
    dates_seq = df.index[LOOKBACK:]

    train_mask = dates_seq < df.index[train_end]
    test_mask  = (dates_seq >= df.index[train_end]) & (dates_seq < df.index[test_end])
    deploy_mask= dates_seq >= df.index[test_end]

    X_train, y_train = X_seq[train_mask], y_seq[train_mask]
    X_test,  y_test  = X_seq[test_mask],  y_seq[test_mask]
    X_deploy, y_deploy = X_seq[deploy_mask], y_seq[deploy_mask]
    dates_deploy = dates_seq[deploy_mask]

    st.success(f"Data ready: {len(df)} days, {len(X_train)} train, {len(X_test)} test, {len(X_deploy)} deploy")

    # --- Train ---
    builder = model_builders[model_name]
    model = builder((LOOKBACK, X_train.shape[2]))
    es = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, min_delta=1e-5)
    rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)

    history = model.fit(
        X_train, y_train,
        validation_split=0.15,
        epochs=60, batch_size=32,
        callbacks=[es, rlr], shuffle=False, verbose=0
    )

    # --- Evaluate on Test ---
    y_test_actual = target_scaler.inverse_transform(y_test.reshape(-1,1)).flatten()
    baseline_pred = np.zeros_like(y_test_actual)

    pred_scaled = model.predict(X_test, verbose=0).flatten()
    pred = target_scaler.inverse_transform(pred_scaled.reshape(-1,1)).flatten()

    rmse = np.sqrt(mean_squared_error(y_test_actual, pred))
    mae  = mean_absolute_error(y_test_actual, pred)
    da   = np.mean(np.sign(y_test_actual) == np.sign(pred)) * 100

    rmse_base = np.sqrt(mean_squared_error(y_test_actual, baseline_pred))
    mae_base  = mean_absolute_error(y_test_actual, baseline_pred)
    da_base   = np.mean(np.sign(y_test_actual) == np.sign(baseline_pred)) * 100

    st.session_state.results = {
        "Model": model_name,
        "Test RMSE": rmse,
        "Test MAE": mae,
        "Test DirAcc": da,
        "Baseline RMSE": rmse_base,
        "Baseline MAE": mae_base,
        "Baseline DirAcc": da_base
    }

    # Tampilkan metrik
    st.subheader("Evaluation on Test Set")
    col1, col2, col3 = st.columns(3)
    col1.metric("Model RMSE", f"{rmse:.6f}")
    col2.metric("Model MAE", f"{mae:.6f}")
    col3.metric("Directional Acc.", f"{da:.1f}%")

    col1, col2, col3 = st.columns(3)
    col1.metric("Baseline RMSE", f"{rmse_base:.6f}")
    col2.metric("Baseline MAE", f"{mae_base:.6f}")
    col3.metric("Baseline DirAcc", f"{da_base:.1f}%")

    # Plot test predictions
    fig, ax = plt.subplots(figsize=(12,5))
    dates_test = dates_seq[test_mask]
    ax.plot(dates_test, y_test_actual, label="Actual", alpha=0.7)
    ax.plot(dates_test, pred, label=f"{model_name} Pred", alpha=0.7)
    ax.plot(dates_test, baseline_pred, label="Baseline (0)", alpha=0.5)
    ax.legend()
    ax.set_title("Test Set: Actual vs Predicted Log Return")
    st.pyplot(fig)

    # --- Deployment Test ---
    # Retrain on all pre-deployment data
    X_pre = np.concatenate([X_train, X_test])
    y_pre = np.concatenate([y_train, y_test])
    model_pre = builder((LOOKBACK, X_pre.shape[2]))
    es2 = EarlyStopping(monitor='loss', patience=7, restore_best_weights=True, min_delta=1e-5)
    model_pre.fit(X_pre, y_pre, epochs=40, batch_size=32,
                  callbacks=[es2], shuffle=False, verbose=0)
    deploy_pred_scaled = model_pre.predict(X_deploy, verbose=0).flatten()
    deploy_pred = target_scaler.inverse_transform(deploy_pred_scaled.reshape(-1,1)).flatten()
    y_deploy_actual = target_scaler.inverse_transform(y_deploy.reshape(-1,1)).flatten()

    deploy_df_display = pd.DataFrame({
        "Date": dates_deploy,
        "Actual": y_deploy_actual,
        "Predicted": deploy_pred,
        "Abs Error": np.abs(y_deploy_actual - deploy_pred)
    })

    st.subheader("Deployment Test (Last 10%)")
    st.dataframe(deploy_df_display.style.format("{:.6f}"))

    # Simpan model jika lebih baik dari baseline
    st.session_state.model = model
    st.session_state.pipeline = {
        "model": model,
        "feat_scaler": feat_scaler,
        "target_scaler": target_scaler,
        "features": features,
        "lookback": LOOKBACK
    }
    st.session_state.trained = True

# Jika sudah pernah train, tampilkan tombol untuk menyimpan model
if st.session_state.trained:
    st.sidebar.success("Model trained!")
    if st.sidebar.button("Save Model (.joblib)"):
        pipeline = st.session_state.pipeline
        joblib.dump(pipeline, "forecast_model.joblib")
        st.sidebar.success("Model saved to forecast_model.joblib")
        with open("forecast_model.joblib", "rb") as f:
            st.sidebar.download_button("Download Model", f, file_name="forecast_model.joblib")

Writing app.py


In [10]:
# Cell 3: Setup ngrok with interactive authtoken and run Streamlit
from pyngrok import ngrok
from getpass import getpass
import subprocess, time, os

# Kill previous tunnels (if any)
ngrok.kill()

# Minta authtoken dari pengguna (input tersembunyi)
print("Masukkan ngrok authtoken Anda (dari https://dashboard.ngrok.com/get-started/your-authtoken)")
token = getpass("Token: ")

# Konfigurasi authtoken
!ngrok authtoken {token}
print("✅ Authtoken telah dikonfigurasi.\n")

# Jalankan Streamlit di port 8501
proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(5)  # Tunggu server siap

# Buka tunnel ke port Streamlit
public_url = ngrok.connect(8501)
print(f"🌐 Dashboard URL: {public_url}")

Masukkan ngrok authtoken Anda (dari https://dashboard.ngrok.com/get-started/your-authtoken)
Token: ··········
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
✅ Authtoken telah dikonfigurasi.

🌐 Dashboard URL: NgrokTunnel: "https://edginess-reunion-marshland.ngrok-free.dev" -> "http://localhost:8501"
